In [1]:
import pandas as pd
import numpy as np
import os

data_path = os.path.expanduser('~/Desktop/football-intelligence/data/')

# Chargement de toutes les données
players = pd.read_csv(data_path + 'players.csv')
valuations = pd.read_csv(data_path + 'player_valuations.csv')
appearances = pd.read_csv(data_path + 'appearances.csv')
clubs = pd.read_csv(data_path + 'clubs.csv')
transfers = pd.read_csv(data_path + 'transfers.csv')
player_profile = pd.read_csv(data_path + 'player_profile_clean.csv')

# Top 5 ligues uniquement
TOP5 = ['GB1', 'ES1', 'FR1', 'IT1', 'L1']
clubs_top5 = clubs[clubs['domestic_competition_id'].isin(TOP5)].copy()
appearances_top5 = appearances[appearances['competition_id'].isin(TOP5)].copy()
players_top5 = player_profile[player_profile['domestic_competition_id'].isin(TOP5)].copy()

print(f"✅ Joueurs Top 5 : {len(players_top5):,}")
print(f"✅ Clubs Top 5 : {len(clubs_top5)}")
print(f"✅ Apparences Top 5 : {len(appearances_top5):,}")
print(f"✅ Valuations : {len(valuations):,}")
print(f"✅ Transferts : {len(transfers):,}")

✅ Joueurs Top 5 : 7,163
✅ Clubs Top 5 : 176
✅ Apparences Top 5 : 726,823
✅ Valuations : 507,815
✅ Transferts : 35,139


In [2]:
# ============================================
# ÉTAPE 1 — TRENDING VALEUR MARCHANDE
# ============================================

# Convertir les dates
valuations['date'] = pd.to_datetime(valuations['date'])

# Trier par joueur et date
valuations_sorted = valuations.sort_values(['player_id', 'date'])

# Pour chaque joueur, calculer la tendance
def calculate_trending(group):
    if len(group) < 2:
        return pd.Series({
            'current_value': group['market_value_in_eur'].iloc[-1],
            'peak_value': group['market_value_in_eur'].max(),
            'value_6m_ago': group['market_value_in_eur'].iloc[-1],
            'value_1y_ago': group['market_value_in_eur'].iloc[-1],
            'trending': 'stable',
            'trending_pct': 0.0
        })
    
    current = group['market_value_in_eur'].iloc[-1]
    peak = group['market_value_in_eur'].max()
    
    # Valeur il y a 6 mois
    six_months_ago = group['date'].iloc[-1] - pd.DateOffset(months=6)
    past_6m = group[group['date'] <= six_months_ago]
    value_6m = past_6m['market_value_in_eur'].iloc[-1] if len(past_6m) > 0 else current
    
    # Valeur il y a 1 an
    one_year_ago = group['date'].iloc[-1] - pd.DateOffset(months=12)
    past_1y = group[group['date'] <= one_year_ago]
    value_1y = past_1y['market_value_in_eur'].iloc[-1] if len(past_1y) > 0 else current
    
    # Calcul tendance
    trending_pct = ((current - value_6m) / (value_6m + 1)) * 100
    
    if trending_pct > 10:
        trending = 'hausse'
    elif trending_pct < -10:
        trending = 'baisse'
    else:
        trending = 'stable'
    
    return pd.Series({
        'current_value': current,
        'peak_value': peak,
        'value_6m_ago': value_6m,
        'value_1y_ago': value_1y,
        'trending': trending,
        'trending_pct': round(trending_pct, 1)
    })

print("⏳ Calcul des tendances en cours...")
player_trending = valuations_sorted.groupby('player_id').apply(
    calculate_trending, include_groups=False
).reset_index()

print(f"✅ Tendances calculées pour {len(player_trending):,} joueurs")
print(f"\n📈 En hausse : {(player_trending['trending'] == 'hausse').sum():,}")
print(f"📉 En baisse : {(player_trending['trending'] == 'baisse').sum():,}")
print(f"➡️ Stable : {(player_trending['trending'] == 'stable').sum():,}")

⏳ Calcul des tendances en cours...
✅ Tendances calculées pour 31,507 joueurs

📈 En hausse : 5,671
📉 En baisse : 15,486
➡️ Stable : 10,350


In [3]:
# ============================================
# ÉTAPE 2 — BESOINS DES CLUBS PAR POSTE
# ============================================

# Compter les joueurs par club et par poste
club_position_count = players_top5.groupby(
    ['current_club_id', 'position']
).size().reset_index(name='player_count')

# Effectif moyen par poste dans les Top 5 ligues
avg_by_position = club_position_count.groupby('position')['player_count'].mean().round(1)
print("=== Effectif moyen par poste ===")
print(avg_by_position)

# Identifier les clubs qui manquent de joueurs à un poste
def get_club_needs(club_id):
    club_data = club_position_count[club_position_count['current_club_id'] == club_id]
    needs = {}
    for position in ['Attack', 'Midfield', 'Defender', 'Goalkeeper']:
        current = club_data[club_data['position'] == position]['player_count'].sum()
        avg = avg_by_position.get(position, 5)
        deficit = avg - current
        needs[position] = {
            'current': int(current),
            'average': float(avg),
            'deficit': round(float(deficit), 1),
            'needs_player': deficit > 1.0
        }
    return needs

# Créer un profil de besoins pour chaque club
club_needs = {}
for club_id in clubs_top5['club_id'].unique():
    club_needs[club_id] = get_club_needs(club_id)

print(f"\n✅ Profils de besoins calculés pour {len(club_needs)} clubs")

# Exemple — besoins du Real Madrid
real_madrid = clubs_top5[clubs_top5['name'].str.contains('Real Madrid')]['club_id'].values
if len(real_madrid) > 0:
    print(f"\n=== Besoins Real Madrid ===")
    for pos, data in club_needs[real_madrid[0]].items():
        status = "⚠️ BESOIN" if data['needs_player'] else "✅ OK"
        print(f"{pos}: {data['current']} joueurs (moy: {data['average']}) {status}")

=== Effectif moyen par poste ===
position
Attack        11.5
Defender      13.6
Goalkeeper     3.9
Midfield      12.1
Missing        1.0
Name: player_count, dtype: float64

✅ Profils de besoins calculés pour 176 clubs

=== Besoins Real Madrid ===
Attack: 8 joueurs (moy: 11.5) ⚠️ BESOIN
Midfield: 8 joueurs (moy: 12.1) ⚠️ BESOIN
Defender: 11 joueurs (moy: 13.6) ⚠️ BESOIN
Goalkeeper: 3 joueurs (moy: 3.9) ✅ OK


In [4]:
# ============================================
# ÉTAPE 3 — STYLE DE JEU DES CLUBS
# ============================================

# Calculer le style de jeu de chaque club à partir des stats de ses joueurs
club_style = players_top5.groupby('current_club_id').agg(
    avg_goals_per90=('goals_per90', 'mean'),
    avg_assists_per90=('assists_per90', 'mean'),
    avg_minutes_per_game=('minutes_per_game', 'mean'),
    avg_age=('age', 'mean'),
    total_players=('player_id', 'count'),
    avg_market_value=('market_value_in_eur', 'mean'),
    total_market_value=('market_value_in_eur', 'sum')
).reset_index()

# Style offensif du club
club_style['club_offensive_style'] = (
    club_style['avg_goals_per90'] * 0.6 +
    club_style['avg_assists_per90'] * 0.4
).round(3)

# Style de rotation (minutes moyennes élevées = peu de rotation)
club_style['club_rotation_style'] = (
    club_style['avg_minutes_per_game'] / 90
).clip(0, 1).round(3)

# Profil jeune vs expérimenté
club_style['club_age_profile'] = club_style['avg_age'].round(1)

# Fusion avec infos clubs
club_style = club_style.merge(
    clubs_top5[['club_id', 'name', 'domestic_competition_id', 
                'foreigners_percentage', 'squad_size']],
    left_on='current_club_id', right_on='club_id', how='left'
)

print("✅ Style de jeu calculé pour tous les clubs !")
print(f"\n=== TOP 10 CLUBS LES PLUS OFFENSIFS ===")
print(club_style.nlargest(10, 'club_offensive_style')[
    ['name', 'domestic_competition_id', 'club_offensive_style', 
     'avg_age', 'total_market_value']
].to_string())

✅ Style de jeu calculé pour tous les clubs !

=== TOP 10 CLUBS LES PLUS OFFENSIFS ===
                               name domestic_competition_id  club_offensive_style    avg_age  total_market_value
155                      UD Almería                     ES1                 0.511  34.250000        4.735000e+07
136                       Amiens SC                     FR1                 0.326  35.086957        8.150000e+06
29                    VfL Wolfsburg                      L1                 0.257  29.796296        2.523750e+08
144  1. Fußballclub Heidenheim 1846                      L1                 0.249  27.484848        7.560000e+07
6                Atlético de Madrid                     ES1                 0.179  30.166667        6.006750e+08
66                      Real Madrid                     ES1                 0.172  27.366667        1.303500e+09
23                       Hertha BSC                      L1                 0.170  32.157895        2.043500e+07
134       

In [5]:
# Ajouter offensive_score à players_top5
players_top5['offensive_score'] = (
    players_top5['goals_per90'] * 0.6 + 
    players_top5['assists_per90'] * 0.4
).round(3)

print("✅ offensive_score ajouté !")
print(players_top5['offensive_score'].head(3))

✅ offensive_score ajouté !
0    0.418
1    0.000
3    0.000
Name: offensive_score, dtype: float64


In [6]:
# ============================================
# ÉTAPE 4 — MODÈLE AVANCÉ COMPLET
# ============================================
from sklearn.preprocessing import StandardScaler
from sklearn.metrics.pairwise import cosine_similarity
import pickle

def recommend_clubs_advanced(player_id=None, player_name=None, top_n=5):
    """
    Recommandation avancée basée sur :
    - Style de jeu du joueur vs club
    - Performances récentes + trending
    - Besoin du club au poste du joueur
    - Compatibilité valeur marchande
    """
    # Trouver le joueur
    if player_name:
        player_data = players_top5[players_top5['name_player'].str.lower().str.contains(player_name.lower())]
        if player_data.empty:
            print(f"❌ Joueur '{player_name}' non trouvé.")
            return None
        player_data = player_data.iloc[0]
    else:
        player_data = players_top5[players_top5['player_id'] == player_id].iloc[0]

    pid = player_data['player_id']
    position = player_data['position']
    player_value = player_data['market_value_in_eur']
    current_club = player_data['current_club_id']

    # Trending du joueur
    trending_data = player_trending[player_trending['player_id'] == pid]
    if len(trending_data) > 0:
        trending = trending_data.iloc[0]['trending']
        trending_pct = trending_data.iloc[0]['trending_pct']
    else:
        trending = 'stable'
        trending_pct = 0.0

    # Score pour chaque club
    club_scores = []
    for _, club in club_style.iterrows():
        club_id = club['current_club_id']
        if club_id == current_club:
            continue

        # 1. Compatibilité style de jeu (30%)
        style_diff = abs(player_data['offensive_score'] - club['club_offensive_style'])
        style_score = max(0, 1 - style_diff * 2)

        # 2. Besoin du club au poste (25%)
        needs = club_needs.get(club_id, {})
        position_need = needs.get(position, {})
        need_score = 1.0 if position_need.get('needs_player', False) else 0.3

        # 3. Compatibilité valeur marchande (25%)
        club_avg_value = club['avg_market_value']
        value_ratio = player_value / (club_avg_value + 1)
        value_score = max(0, 1 - abs(1 - value_ratio))

        # 4. Trending joueur (20%)
        if trending == 'hausse':
            trend_score = 1.0
        elif trending == 'stable':
            trend_score = 0.6
        else:
            trend_score = 0.3

        # Score final pondéré
        final_score = (
            style_score * 0.30 +
            need_score * 0.25 +
            value_score * 0.25 +
            trend_score * 0.20
        )

        club_scores.append({
            'club_id': club_id,
            'name': club['name'],
            'league': club['domestic_competition_id'],
            'final_score': round(final_score, 3),
            'style_score': round(style_score * 100, 1),
            'need_score': round(need_score * 100, 1),
            'value_score': round(value_score * 100, 1),
            'trend_score': round(trend_score * 100, 1),
            'needs_player': position_need.get('needs_player', False)
        })

    # Trier + diversifier les ligues
    results_df = pd.DataFrame(club_scores).sort_values('final_score', ascending=False)
    results = []
    league_count = {}
    for _, row in results_df.iterrows():
        league = row['league']
        league_count[league] = league_count.get(league, 0) + 1
        if league_count[league] <= 2:
            results.append(row)
        if len(results) == top_n:
            break

    # Affichage
    print(f"\n🏆 TOP {top_n} CLUBS — {player_data['name_player']}")
    print(f"   Poste : {position} | Valeur : {player_value/1e6:.1f}M€ | Trending : {trending} ({trending_pct:+.1f}%)")
    print("-" * 70)
    for i, row in enumerate(results):
        need = "⚠️ BESOIN" if row['needs_player'] else "✅"
        print(f"  {i+1}. {row['name']} ({row['league']}) — Score: {row['final_score']*100:.1f}%")
        print(f"     Style:{row['style_score']}% | Besoin:{row['need_score']}% | Valeur:{row['value_score']}% | Trend:{row['trend_score']}% {need}")

    return results

# TESTS
recommend_clubs_advanced(player_name="Kylian Mbappé")


🏆 TOP 5 CLUBS — Kylian Mbappé
   Poste : Attack | Valeur : 200.0M€ | Trending : stable (+0.0%)
----------------------------------------------------------------------
  1. UD Almería (ES1) — Score: 56.4%
     Style:64.6% | Besoin:100.0% | Valeur:0% | Trend:60.0% ⚠️ BESOIN
  2. Amiens SC (FR1) — Score: 45.3%
     Style:27.6% | Besoin:100.0% | Valeur:0% | Trend:60.0% ⚠️ BESOIN
  3. 1.FC Nuremberg (L1) — Score: 37.0%
     Style:0.0% | Besoin:100.0% | Valeur:0% | Trend:60.0% ⚠️ BESOIN
  4. Arsenal FC (GB1) — Score: 37.0%
     Style:0.0% | Besoin:100.0% | Valeur:0% | Trend:60.0% ⚠️ BESOIN
  5. Brescia Calcio (IT1) — Score: 37.0%
     Style:0.0% | Besoin:100.0% | Valeur:0% | Trend:60.0% ⚠️ BESOIN


[club_id               3302
 name            UD Almería
 league                 ES1
 final_score          0.564
 style_score           64.6
 need_score           100.0
 value_score              0
 trend_score           60.0
 needs_player          True
 Name: 154, dtype: object,
 club_id              1416
 name            Amiens SC
 league                FR1
 final_score         0.453
 style_score          27.6
 need_score          100.0
 value_score             0
 trend_score          60.0
 needs_player         True
 Name: 135, dtype: object,
 club_id                      4
 name            1.FC Nuremberg
 league                      L1
 final_score               0.37
 style_score                0.0
 need_score               100.0
 value_score                  0
 trend_score               60.0
 needs_player              True
 Name: 1, dtype: object,
 club_id                 11
 name            Arsenal FC
 league                 GB1
 final_score           0.37
 style_score            0.

In [7]:
# Ajouter les scores manquants à players_top5
players_top5 = players_top5.copy()

# Score offensif
players_top5['offensive_score'] = (
    players_top5['goals_per90'] * 0.6 + 
    players_top5['assists_per90'] * 0.4
).round(3)

# Vérification
print("✅ Colonnes disponibles :")
print([c for c in players_top5.columns if 'score' in c.lower()])
print(f"\nExemple offensive_score : {players_top5['offensive_score'].head(3).values}")

✅ Colonnes disponibles :
['offensive_score']

Exemple offensive_score : [0.418 0.    0.   ]


In [8]:
# ============================================
# CORRECTION MODÈLE AVANCÉ V2
# ============================================

def recommend_clubs_advanced_v2(player_name, top_n=5):
    
    player_data = players_top5[players_top5['name_player'].str.lower().str.contains(player_name.lower())]
    if player_data.empty:
        print(f"❌ Joueur non trouvé.")
        return None
    player_data = player_data.iloc[0]

    pid = player_data['player_id']
    position = player_data['position']
    player_value = player_data['market_value_in_eur']
    current_club = player_data['current_club_id']

    # Trending
    trending_data = player_trending[player_trending['player_id'] == pid]
    trending = trending_data.iloc[0]['trending'] if len(trending_data) > 0 else 'stable'
    trending_pct = trending_data.iloc[0]['trending_pct'] if len(trending_data) > 0 else 0.0

    club_scores = []
    for _, club in club_style.iterrows():
        club_id = club['current_club_id']
        if club_id == current_club:
            continue

        # 1. Style de jeu (30%) — similarité offensive
        player_off = player_data['offensive_score']
        club_off = club['club_offensive_style']
        max_off = players_top5['offensive_score'].max()
        style_score = 1 - abs(player_off - club_off) / (max_off + 0.001)
        style_score = max(0, min(1, style_score))

        # 2. Besoin du club au poste (25%)
        needs = club_needs.get(club_id, {})
        position_need = needs.get(position, {})
        need_score = 1.0 if position_need.get('needs_player', False) else 0.4

        # 3. Compatibilité valeur marchande (30%)
        club_total_value = club['total_market_value']
        # Un joueur à 200M€ doit aller dans un club riche
        # Ratio : valeur joueur / (valeur totale effectif / nb joueurs)
        club_avg = club['avg_market_value']
        if club_avg > 0:
            ratio = player_value / club_avg
            # Idéal : ratio entre 0.5 et 3 (joueur dans la moyenne haute du club)
            if 0.5 <= ratio <= 3:
                value_score = 1.0
            elif ratio < 0.5:
                value_score = ratio / 0.5
            else:
                value_score = max(0, 1 - (ratio - 3) / 10)
        else:
            value_score = 0

        # 4. Trending (15%)
        trend_map = {'hausse': 1.0, 'stable': 0.6, 'baisse': 0.3}
        trend_score = trend_map.get(trending, 0.6)

        final_score = (
            style_score * 0.30 +
            need_score * 0.25 +
            value_score * 0.30 +
            trend_score * 0.15
        )

        club_scores.append({
            'club_id': club_id,
            'name': club['name'],
            'league': club['domestic_competition_id'],
            'final_score': round(final_score, 3),
            'style_score': round(style_score * 100, 1),
            'need_score': round(need_score * 100, 1),
            'value_score': round(value_score * 100, 1),
            'trend_score': round(trend_score * 100, 1),
            'needs_player': position_need.get('needs_player', False),
            'club_avg_value': round(club_avg / 1e6, 1)
        })

    # Tri + diversité ligues
    results_df = pd.DataFrame(club_scores).sort_values('final_score', ascending=False)
    results = []
    league_count = {}
    for _, row in results_df.iterrows():
        league = row['league']
        league_count[league] = league_count.get(league, 0) + 1
        if league_count[league] <= 2:
            results.append(row)
        if len(results) == top_n:
            break

    # Affichage
    league_names_map = {
        'GB1': 'Premier League', 'ES1': 'La Liga',
        'FR1': 'Ligue 1', 'IT1': 'Serie A', 'L1': 'Bundesliga'
    }
    trend_icon = '📈' if trending == 'hausse' else ('📉' if trending == 'baisse' else '➡️')

    print(f"\n🏆 TOP {top_n} CLUBS — {player_data['name_player']}")
    print(f"   Poste : {position} | Valeur : {player_value/1e6:.1f}M€ | {trend_icon} Trending : {trending} ({trending_pct:+.1f}%)")
    print("-" * 75)
    for i, row in enumerate(results):
        need = "⚠️ BESOIN AU POSTE" if row['needs_player'] else "✅ Poste OK"
        league = league_names_map.get(row['league'], row['league'])
        print(f"  {i+1}. {row['name']} ({league}) — Score: {row['final_score']*100:.1f}%")
        print(f"     🎨 Style:{row['style_score']}% | 📋 Besoin:{row['need_score']}% | 💰 Valeur:{row['value_score']}% | {trend_icon} Trend:{row['trend_score']}% | {need}")
        print(f"     Valeur moy. club : {row['club_avg_value']}M€")

    return results

# TESTS
print("=" * 75)
recommend_clubs_advanced_v2("Kylian Mbappé")
print("\n" + "=" * 75)
recommend_clubs_advanced_v2("Florian Thauvin")
print("\n" + "=" * 75)
recommend_clubs_advanced_v2("Virgil van Dijk")


🏆 TOP 5 CLUBS — Kylian Mbappé
   Poste : Attack | Valeur : 200.0M€ | ➡️ Trending : stable (+0.0%)
---------------------------------------------------------------------------
  1. Manchester City (Premier League) — Score: 88.3%
     🎨 Style:97.0% | 📋 Besoin:100.0% | 💰 Valeur:84.0% | ➡️ Trend:60.0% | ⚠️ BESOIN AU POSTE
     Valeur moy. club : 43.4M€
  2. Arsenal FC (Premier League) — Score: 86.4%
     🎨 Style:96.9% | 📋 Besoin:100.0% | 💰 Valeur:77.9% | ➡️ Trend:60.0% | ⚠️ BESOIN AU POSTE
     Valeur moy. club : 38.4M€
  3. FC Barcelona (La Liga) — Score: 83.5%
     🎨 Style:97.0% | 📋 Besoin:100.0% | 💰 Valeur:68.0% | ➡️ Trend:60.0% | ⚠️ BESOIN AU POSTE
     Valeur moy. club : 32.2M€
  4. Paris Saint-Germain (Ligue 1) — Score: 83.4%
     🎨 Style:96.9% | 📋 Besoin:100.0% | 💰 Valeur:67.8% | ➡️ Trend:60.0% | ⚠️ BESOIN AU POSTE
     Valeur moy. club : 32.2M€
  5. Bayern Munich (Bundesliga) — Score: 82.1%
     🎨 Style:97.0% | 📋 Besoin:100.0% | 💰 Valeur:63.3% | ➡️ Trend:60.0% | ⚠️ BESOIN AU POSTE


[club_id                               543
 name              Wolverhampton Wanderers
 league                                GB1
 final_score                         0.895
 style_score                         100.0
 need_score                          100.0
 value_score                         100.0
 trend_score                          30.0
 needs_player                         True
 club_avg_value                        9.3
 Name: 72, dtype: object,
 club_id                23826
 name              RB Leipzig
 league                    L1
 final_score            0.895
 style_score             99.9
 need_score             100.0
 value_score            100.0
 trend_score             30.0
 needs_player            True
 club_avg_value          12.8
 Name: 174, dtype: object,
 club_id                            15
 name              Bayer 04 Leverkusen
 league                             L1
 final_score                     0.894
 style_score                      99.7
 need_score           

In [9]:
# Sauvegarde du modèle avancé
import pickle

advanced_model = {
    'players_top5': players_top5,
    'clubs_top5': clubs_top5,
    'club_style': club_style,
    'club_needs': club_needs,
    'player_trending': player_trending,
}

with open(data_path + 'advanced_model.pkl', 'wb') as f:
    pickle.dump(advanced_model, f)

print("✅ Modèle avancé sauvegardé !")
print(f"   - {len(players_top5):,} joueurs Top 5")
print(f"   - {len(clubs_top5)} clubs")
print(f"   - {len(player_trending):,} tendances calculées")
print(f"   - {len(club_needs)} profils de besoins clubs")

✅ Modèle avancé sauvegardé !
   - 7,163 joueurs Top 5
   - 176 clubs
   - 31,507 tendances calculées
   - 176 profils de besoins clubs


In [10]:
# Chercher des joueurs disponibles pour tester
test_names = ['Modric', 'Haaland', 'Endrick', 'Vinicius', 'Bellingham']
for name in test_names:
    found = players_top5[players_top5['name_player'].str.lower().str.contains(name.lower())]
    if len(found) > 0:
        print(f"✅ {name} → {found.iloc[0]['name_player']} ({found.iloc[0]['name_club']})")
    else:
        print(f"❌ {name} → non trouvé")

❌ Modric → non trouvé
✅ Haaland → Erling Haaland (Manchester City Football Club)
✅ Endrick → Hendrick Zuck (Sport-Club Freiburg)
✅ Vinicius → Vinicius Junior (Real Madrid Club de Fútbol)
✅ Bellingham → Jude Bellingham (Real Madrid Club de Fútbol)
